# Day 6: Preference tuning

Yesterday's model answers, but its greedy answers repeat themselves and often never end. Today it is shown pairs of
answers to the same question, one better than the other, and learns to prefer the better kind.

This notebook runs [`scripts/train_dpo.py`](https://github.com/a-ramakrishnan/mini_llm/blob/main/scripts/train_dpo.py).
It needs Day 5's run on your Google Drive (`mini_llm/runs/<preset>-sft`).

## Set up

**Runtime → Change runtime type → a GPU.** A free T4 runs the mini path. The full runs were done on an L4 and an A100.

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU (or better)"
print(torch.cuda.get_device_name(0))

Runs are saved to Google Drive, so a disconnected session loses nothing: run the notebook again and training carries
on from the last saved state. Token files are kept on the fast local disk and backed up to Drive.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
WORK = "/content/drive/MyDrive/mini_llm"      # runs/ and a copy of the token files live here
os.makedirs(f"{WORK}/runs", exist_ok=True)
os.makedirs(f"{WORK}/data", exist_ok=True)
os.environ["MINI_LLM_DATA"] = "/content/data"  # token files are read from the local disk: Drive is too slow
os.environ["WORK"] = WORK

In [ ]:
%cd /content
!test -d mini_llm || git clone -q https://github.com/a-ramakrishnan/mini_llm.git
%cd /content/mini_llm
!git pull -q
!pip install -q tiktoken datasets

In [ ]:
PRESET = "mini"  # "mini": 29.9M parameters, the mini path. "full": the shipped 124M run.
os.environ["PRESET"] = PRESET

In [ ]:
SFT = f"{WORK}/runs/{PRESET}-sft"
RUN = f"{WORK}/runs/{PRESET}-dpo"
PAIRS = f"{WORK}/data/dpo-pairs-{PRESET}.jsonl"     # on Drive: making the pairs is the slow part
assert os.path.exists(f"{SFT}/best_model_params.pt"), "Run Day 5 first: this tunes its model"

## 1. Make the pairs

For each question, two answers:

- **chosen:** the reference answer from smol-smoltalk. Only questions with a short reference answer are used, so the
  chosen answer always ends.
- **rejected:** the model's own greedy answer to the same question, loops and all.

Nobody labels anything. The file is appended to as it goes, so an interrupted run carries on.

In [ ]:
!python scripts/train_dpo.py --preset {PRESET} --model "{SFT}" --pairs "{PAIRS}" --out "{RUN}" --only pairs

In [ ]:
import json
from mini_llm.tokenizer import get_tokenizer
enc = get_tokenizer()
pairs = [json.loads(l) for l in open(PAIRS)]
p = max(pairs[:200], key=lambda p: p["rejected_rep4"])
print(f"{len(pairs):,} pairs; {sum(not p['rejected_ended'] for p in pairs) / len(pairs):.0%} of the rejected answers never end
")
print("QUESTION:", p["question"], "

CHOSEN:", p["chosen"], "

REJECTED:", enc.decode(p["rejected_ids"]).strip())

## 2. Train

DPO raises the probability of the chosen answer and lowers the rejected one's, both measured against a **frozen copy
of the starting model**:

`loss = -log sigmoid( beta x [ (log p(chosen) - log p_ref(chosen)) - (log p(rejected) - log p_ref(rejected)) ] )`

- At the start the model equals its frozen copy, so the bracket is 0 and the loss is log 2 = 0.693.
- The learning rate is 2e-6, a hundredth of instruction tuning's: a nudge, not a retraining.
- A small likelihood term on the chosen answer keeps DPO from lowering both answers' probability.

In [ ]:
!python scripts/train_dpo.py --preset {PRESET} --model "{SFT}" --pairs "{PAIRS}" --out "{RUN}" --only train

## 3. Measure, on questions training never saw

Both models answer the held-out questions greedily: how much they repeat, how many answers end by themselves, and the
loss on the reference answers (did the model pay for it in answer quality?).

In [ ]:
!python scripts/train_dpo.py --preset {PRESET} --model "{SFT}" --pairs "{PAIRS}" --out "{RUN}" --only eval

In [ ]:
import pandas as pd
r = json.load(open(f"{RUN}/dpo_eval.json"))["results"]
pd.DataFrame({k: {"repeated 4-token runs": f"{v['repeated_4grams']:.1%}", "answers that end": f"{v['ended']:.0%}",
                  "mean length (tokens)": v["mean_length"], "loss on reference answers": v["reference_loss"]}
              for k, v in r.items()})

## 4. The same questions, before and after

In [ ]:
for run in (SFT, RUN):
    print("=" * 20, os.path.basename(run))
    !python scripts/chat.py "What is the difference between weather and climate?" --model "{run}"

That is the last training stage. `mini_llm/runs/<preset>-dpo` on your Drive is the finished chat model:
`best_model_params.pt` and `model_config.json`. Days 7 and 8 measure it.